In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Data Transformations
--- 

#### Create Schema

In [0]:
spark.sql('create schema if not exists test01.silver')

---

#### cust_info

In [0]:
w = Window.partitionBy('cst_id').orderBy(col('cst_create_date').desc())

df_crm_cust_info = (
    spark.table('test01.bronze.crm_cust_info')
    .filter(col('cst_id').isNotNull())
    .withColumn('flag_last', row_number().over(w))
    .filter(col('flag_last') == 1)
    .drop('flag_last')
    .withColumn(
        'cst_marital_status', when(upper(col('cst_marital_status')) == 'M', 'Marriage')
        .when(upper(col('cst_marital_status')) == 'S', 'Single')
        .otherwise('n/a')
    )
    .withColumn(
        'cst_gndr', when(upper(col('cst_gndr')) == 'M', 'Male')
        .when(upper(col('cst_gndr')) == 'F', 'Female')
        .otherwise('n/a')
    )
    .withColumn(
        'cst_firstname', trim(col('cst_firstname'))
    )
    .withColumn(
        'cst_lastname', trim(col('cst_lastname'))
    )
)

df_crm_cust_info.display()

#### check

#### Check if it has duplicate cst_id or not ( should see nothing )

In [0]:
df_crm_cust_info.groupBy(col('cst_id')).agg(count('*')).filter(col('count(1)') > 1).display()

####check it has only 2 marital_status or not (Marriage, Single)

In [0]:
df_crm_cust_info.groupBy(col('cst_marital_status')).agg(count('*')).display()

#### check if it has only 3 gndr or not (Male, Female, n/a)

In [0]:
df_crm_cust_info.groupBy(col('cst_gndr')).agg(count('*')).display()

---
### prd_info

In [0]:
w_prd = Window.partitionBy('prd_key').orderBy(col('prd_start_dt'))

df_crm_prd_info = (
    spark.table('test01.bronze.crm_prd_info')
    .filter(col('prd_id').isNotNull())
    .withColumn(
        'cat_id', regexp_replace(substring(col('prd_key'), 1, 5), '-', '_')
    )
    .withColumn(
        'prd_key', substring(col('prd_key'), 7, length('prd_key'))
    )
    .withColumn(
        'prd_cost', coalesce(col('prd_cost'), lit(0))
    )
    .withColumn(
        'prd_line', when(upper(trim(col('prd_line'))) == 'R', 'Road')
        .when(upper(trim(col('prd_line'))) == 'M', 'Mountain')
        .when(upper(trim(col('prd_line'))) == 'S', 'Other Sales')
        .when(upper(trim(col('prd_line'))) == 'T', 'Touring')
        .otherwise('n/a')
    )
    .withColumn(
        'prd_end_dt', date_sub(lead('prd_start_dt', 1).over(w_prd), 1)
    )
)

df_crm_prd_info.sort('prd_key').display()

---
### Sales Details

In [0]:
df_crm_sales_details = (
    spark.table('test01.bronze.crm_sales_details')
    .withColumn(
        'sls_order_dt', when((col('sls_order_dt') == 0) | (length(col('sls_order_dt')) != 8), lit(None))
        .otherwise(to_date(col('sls_order_dt'), 'yyyyMMdd'))
    )
    .withColumn(
        'sls_due_dt', when((col('sls_due_dt') == 0) | (length(col('sls_due_dt')) != 8), lit(None))
        .otherwise(to_date(col('sls_due_dt'), 'yyyyMMdd'))
    )
    .withColumn(
        'sls_ship_dt', when((col('sls_ship_dt') == 0) | (length(col('sls_ship_dt')) != 8), lit(None))
        .otherwise(to_date(col('sls_ship_dt'), 'yyyyMMdd'))
    )
    .withColumn(
        'sls_sales', when((col('sls_sales') <= 0) | (col('sls_sales').isNull()) | (col('sls_sales') != col('sls_quantity') * abs(col('sls_price'))), col('sls_quantity') * abs(col('sls_price')))
        .otherwise(col('sls_sales'))
    )
    .withColumn(
        'sls_price', when((col('sls_price') <= 0) | (col('sls_price').isNull()), col('sls_sales') / col('sls_quantity'))
        .otherwise(col('sls_price'))
    )
)

df_crm_sales_details.display()

---
### cust_az12

In [0]:
df_erp_cust_az12 = (
    spark.table('test01.bronze.erp_cust_az12')
    .withColumn(
        'cid', when(col('cid').startswith('NAS'), substring(col('cid'), 4, length(col('cid'))))
        .otherwise(col('cid'))
    )
    .withColumn(
        'bdate', when((col('bdate') > current_date()) | (col('bdate') < '1926-01-01'), lit(None))
        .otherwise(col('bdate'))
    )
    .withColumn(
        'gen', when(upper(trim(col('gen'))).isin('F', 'FEMALE'), 'Female')
        .when(upper(trim(col('gen'))).isin('M', 'MALE'), 'Male')
        .otherwise('n/a')
    )
)

df_erp_cust_az12.display()

---
### loc_a101

In [0]:
df_erp_loc_a101 = (
    spark.table('test01.bronze.erp_loc_a101')
    .withColumn(
        'cid', regexp_replace(col('cid'), '-', '')
    )
    .withColumn(
        'cntry', when(trim(col('cntry')).isin('US', 'United States', 'USA'), 'United States')
        .when(trim(col('cntry')).isin('DE', 'Germany'), 'Germany')
        .when((col('cntry').isNull()) | (trim(col('cntry')) == ''), 'n/a')
        .otherwise(trim(col('cntry')))
    )
)

df_erp_loc_a101.display()

---
### px_cat_g1v2

In [0]:
df_erp_px_cat_g1v2 = (
    spark.table('test01.bronze.erp_px_cat_g1v2')
)

df_erp_px_cat_g1v2.display()

---
### Write

In [0]:
df_crm_cust_info.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.crm_cust_info')

spark.table('test01.silver.crm_cust_info').display()

In [0]:
df_crm_prd_info.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.crm_prd_info')

spark.table('test01.silver.crm_prd_info').display()

In [0]:
df_crm_sales_details.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.crm_sales_details')

spark.table('test01.silver.crm_sales_details').display()

In [0]:
df_erp_cust_az12.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.erp_cust_az12')

spark.table('test01.silver.erp_cust_az12').display()

In [0]:
df_erp_loc_a101.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.erp_loc_a101')

spark.table('test01.silver.erp_loc_a101').display()

In [0]:
df_erp_px_cat_g1v2.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.silver.erp_px_cat_g1v2')

spark.table('test01.silver.erp_px_cat_g1v2').display()